<a href="https://colab.research.google.com/github/amita-kapoor/Agentic-Systems-Engineering/blob/main/Chapter03/CrewAI_booking_a_flight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install CrewAI
#pip install -U crewai
import os

os.environ["OPENAI_API_KEY"] = "Your-API KEY"  # Replace with your key


In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

# Define LLM for agents
llm = LLM(model="gpt-3.5-turbo")  # You can use "gpt-4" if desired

# Define agents
planner = Agent(
    role="Travel Planner",
    goal="Decompose the user's flight booking task into meaningful subtasks and route it to the correct agent.",
    backstory="You're a smart travel planner who understands how to break down complex flight booking tasks and direct them appropriately.",
    llm=llm,
    verbose=True
)

domestic_agent = Agent(
    role="Domestic Flight Booker",
    goal="Find the best flight options within the user's home country based on user preferences.",
    backstory="You specialize in domestic flight bookings and are known for efficient and affordable recommendations.",
    llm=llm,
    verbose=True
)

international_agent = Agent(
    role="International Flight Booker",
    goal="Find the best international flights within budget and help the user prepare documents.",
    backstory="You are an expert in global travel logistics and booking cost-effective international flights.",
    llm=llm,
    verbose=True
)

# Define tasks
planner_task = Task(
    description=(
        "Decide whether the destination '{destination}' is domestic or international "
        "and forward the booking task to the right agent. Then summarize what happens."
    ),
    expected_output="An overview of which agent was selected and what steps were taken.",
    agent=planner
)

domestic_task = Task(
    description=(
        "Search for the best domestic flights to '{destination}' from '{departure_city}' "
        "under ${budget} and prepare one option with airline, timing, and price."
    ),
    expected_output="One domestic flight option with airline, time, and cost.",
    agent=domestic_agent
)

international_task = Task(
    description=(
        "Search international flights to '{destination}' from '{departure_city}' under ${budget}, "
        "and verify required travel documents. Provide at least one option."
    ),
    expected_output="One international flight option and document checklist.",
    agent=international_agent
)

# Set up the crew
crew = Crew(
    agents=[planner, domestic_agent, international_agent],
    tasks=[planner_task, domestic_task, international_task],
    process=Process.sequential,  # Optional: you could use Process.hierarchical later
    verbose=True
)

# Provide sample input
inputs = {
    "destination": "Paris",
    "departure_city": "New York",
    "budget": 800
}

# Run the crew
result = crew.kickoff(inputs=inputs)
print("\n✅ Final Output:\n", result.raw)